In [13]:
# Torch version
!python -c "import torch; print(torch.__version__)"

# Cuda version
!python -c "import torch; print(torch.version.cuda)"

2.6.0+cu124


12.4


In [14]:
from model_PyG import *
from utils import *

In [15]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torch_geometric.transforms as T
import os
import umap

from mpl_toolkits.mplot3d import Axes3D
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from torch_geometric.data import InMemoryDataset, Data
from torch_geometric.utils import dense_to_sparse, negative_sampling
from torch.nn.functional import binary_cross_entropy_with_logits
from torch.optim import Adam

In [16]:
import torch_geometric
print(torch_geometric.__version__)

2.7.0


### Utils

In [17]:
def info(data):
	print("Validate:\t {}".format(data.validate(raise_on_error=True)))
	print("Num. nodes:\t {}".format(data.num_nodes))
	print("Num. edges:\t {}".format(data.num_edges))
	print("Num. features:\t {}".format(data.num_node_features))
	print("Has isolated:\t {}".format(data.has_isolated_nodes()))
	print("Has loops:\t {}".format(data.has_self_loops()))
	print("Is directed:\t {}".format(data.is_directed()))
	print("Is undirected:\t {}".format(data.is_undirected()))
	print("{}".format(data.edge_index))
	print("{}".format(data.x))
	print("{}".format(data.edge_attr))

def compute_num_neg_samples(edge_index, num_nodes, ratio):
	E = edge_index.size(1)
	max_neg = num_nodes * num_nodes - E
	return min(int(ratio * E), max_neg)

def neg_ratio_schedule(epoch, max_epoch):
	start = 5.0
	end = 1.0
	return start - (start - end) * (epoch / max_epoch)

class EarlyStopping:
	def __init__(self, patience=5, delta=0, warmup=5, verbose=False):
		self.patience = patience
		self.delta = delta
		self.warmup = warmup
		self.verbose = verbose
		self.best_loss = None
		self.no_improvement_count = 0
		self.stop_training = False
	
	def check_early_stop(self, loss, epoch):
		if epoch >= self.warmup:
			if self.best_loss is None or loss < self.best_loss - self.delta:
				self.best_loss = loss
				self.no_improvement_count = 0
			else:
				self.no_improvement_count += 1
				if self.no_improvement_count >= self.patience:
					self.stop_training = True
					if self.verbose:
						print("Stopping early as no improvement has been observed.")

### Parameters

In [18]:
file = open("exp.json")
experiment = json.load(file)
exp = experiment["exp"] # experiment["exp"] # Change to static, e.g. "exp1"

file = open("experiments/output/{}/parameters.json".format(exp))
params = json.load(file)

print("Exp:\t\t", exp)

raw_data_file = params["raw_data_file"]
print("Raw data:\t", raw_data_file)

methods = params["methods"]
print("Methods:\t", methods)

apply_transformation = params["apply_transformation"]
print("Has transformation:", apply_transformation)

dimension = params["dimension"]
print("Dimension:\t", dimension)

groups_id = params["groups_id"]
print("Groups id:\t", groups_id)

subgroups_id = params["subgroups_id"]
print("Subgroups id:\t", subgroups_id)

# encoders = ["GIN", "GINE"]
encoders = ["GIN"]

Exp:		 exp30
Raw data:	 Pablo_new_2br_mid_ar_p_42_format
Methods:	 ['t-gae']
Has transformation: False
Dimension:	 64
Groups id:	 ['AR', 'BC', 'BPH', 'CKD-Mild', 'CKD-Moderate', 'CKD-Severe', 'CKD5-HD', 'CKD5-PD', 'CRS', 'Health', 'LPRD', 'LSNB', 'OSA', 'PCa', 'PD', 'RCC', 'SGB', 'SKD']
Subgroups id:	 {'AR': ['1', '2'], 'BC': ['1', '2'], 'BPH': ['1', '2'], 'CKD-Mild': ['1', '2'], 'CKD-Moderate': ['1', '2'], 'CKD-Severe': ['1', '2'], 'CKD5-HD': ['1', '2'], 'CKD5-PD': ['1', '2'], 'CRS': ['1', '2'], 'Health': ['1', '2'], 'LPRD': ['1', '2'], 'LSNB': ['1', '2'], 'OSA': ['1', '2'], 'PCa': ['1', '2'], 'PD': ['1', '2'], 'RCC': ['1', '2'], 'SGB': ['1', '2'], 'SKD': ['1', '2']}


In [19]:
# Remove
# groups_id = ["LSNB", "BC", "RCC", "BPH", "PD"] # "OSA" Memory issue
groups_id = ["AR", "Health", "CKD-Mild", "CKD-Severe", "LPRD"]

### Similarity analysis (KNN)

In [20]:
k = 1 # Change

for encoder in encoders:
	for group_id in groups_id:
		df_node_embeddings_concat = pd.read_csv(f"experiments/output/{exp}/node_embeddings/{encoder}_{group_id}.csv", dtype={"subgroup_id": "string"})
		# Id, 0, 1,	2, ..., subgroup_id

		subgroups_id_ = subgroups_id[group_id]

		# Calculate distance matrix (KNN)
		knn = NearestNeighbors(n_neighbors=k, metric="euclidean")

		df_node_embeddings = df_node_embeddings_concat[df_node_embeddings_concat["subgroup_id"] == subgroups_id_[0]]
		x = df_node_embeddings.iloc[:, 1:-1].values

		df_node_alignment = pd.DataFrame()
		df_node_alignment[f"{group_id}_{subgroups_id_[0]}"] = df_node_embeddings["Id"].values

		for subgroup_id in subgroups_id_[1:]:
			df_node_embeddings = df_node_embeddings_concat[df_node_embeddings_concat["subgroup_id"] == subgroup_id]
			y = df_node_embeddings.iloc[:, 1:-1].values

			knn.fit(y)
			distances, indices = knn.kneighbors(x)
			indices = indices.squeeze() # (N,)

			df_node_alignment[f"{group_id}_{subgroup_id}"] = df_node_embeddings["Id"].values[indices]
			
			""" df_node_alignment[f"distances"] = distances
			avg_distances = df_node_alignment["distances"].mean()
			df_node_alignment = df_node_alignment[df_node_alignment["distances"] <= avg_distances].iloc[:, :-1]
			df_node_alignment """
		
		# Find node alignment for all datasets
		df_node_alignment_filter = df_node_alignment[df_node_alignment.nunique(axis=1) == 1]
		# print(len(df_node_alignment_filter))
		# display(df_node_alignment_filter)
		print(f"{encoder}-{group_id}: {len(df_node_alignment_filter)}")

		# Save common node id as .
		# common_node_id = df_node_alignment_filter.iloc[:, 0].values
		# np.save(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.npy", np.array(common_node_id))

		# Save common node id as .csv
		df_common_node_id = df_node_alignment_filter.iloc[:, [0]].copy()
		df_common_node_id.sort_values(by=df_common_node_id.columns[0], inplace=True)
		df_common_node_id.to_csv(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.csv", index=False, header=False)

# Intersection and Union
list_common_node_id = []
for encoder in encoders:
	for group_id in groups_id:
		# Read common node
		# common_node_id = np.load(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.npy")
		common_node_id = pd.read_csv(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.csv", header=None)
		list_common_node_id.append(common_node_id[0].to_numpy())

	common_node_id_union = list(set.union(*map(set, list_common_node_id)))
	common_node_id_intersection = list(set.intersection(*map(set, list_common_node_id)))
	print(f"{encoder}-union: {len(common_node_id_union)}")
	print(f"{encoder}-intersection: {len(common_node_id_intersection)}")

	# Save common node id
	# np.save(f"experiments/output/{exp}/common_nodes/{encoder}_union.npy", np.array(common_node_id_union))
	# np.save(f"experiments/output/{exp}/common_nodes/{encoder}_intersection.npy", np.array(common_node_id_intersection))

	df_common_node_id_union = pd.DataFrame(common_node_id_union)
	df_common_node_id_union.sort_values(by=df_common_node_id_union.columns[0], inplace=True)
	df_common_node_id_intersection = pd.DataFrame(common_node_id_intersection)
	if len(df_common_node_id_intersection) > 0:
		df_common_node_id_intersection.sort_values(by=df_common_node_id_intersection.columns[0], inplace=True)

	df_common_node_id_union.to_csv(f"experiments/output/{exp}/common_nodes/{encoder}_union.csv", index=False, header=False)
	df_common_node_id_intersection.to_csv(f"experiments/output/{exp}/common_nodes/{encoder}_intersection.csv", index=False, header=False)

GIN-AR: 589
GIN-Health: 166
GIN-CKD-Mild: 691
GIN-CKD-Severe: 886
GIN-LPRD: 159
GIN-union: 2011
GIN-intersection: 1


In [21]:
df_common_node_id_union

,0
5,16
7,19
9,20
12,25
14,27
...,...
1020,5439
1021,5440
1022,5441
1023,5442


In [22]:
df_node_alignment

,LPRD_1,LPRD_2
0,0,458
1,19,101
2,43,17
3,46,185
4,51,700
...,...,...
5439,1009,1239
5440,1330,475
5441,2844,3146
5442,3492,3315


### Filter MS data

In [23]:
# Read raw data

""" df_join_raw = pd.read_csv(f"experiments/input/{exp}_raw.csv", index_col=0)
df_join_raw """

' df_join_raw = pd.read_csv(f"experiments/input/{exp}_raw.csv", index_col=0)\ndf_join_raw '

In [24]:
# Filter
""" for encoder in encoders:
	for group_id in groups_id:
		# Read common node
		common_node_id = np.load(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.npy")
		print(f"{encoder}-{group_id}: {len(common_node_id)}/{len(df_join_raw)}")

		# Filter raw data
		df_join_raw_filter = df_join_raw.loc[common_node_id].iloc[:, [0, 1, 2]]
		df_join_raw_filter.to_csv(f"experiments/output/{exp}/filter_raw/{encoder}_{group_id}.csv", sep=";", decimal=",", index_label="Id")
		df_join_raw_filter """

' for encoder in encoders:\n\tfor group_id in groups_id:\n\t\t# Read common node\n\t\tcommon_node_id = np.load(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.npy")\n\t\tprint(f"{encoder}-{group_id}: {len(common_node_id)}/{len(df_join_raw)}")\n\n\t\t# Filter raw data\n\t\tdf_join_raw_filter = df_join_raw.loc[common_node_id].iloc[:, [0, 1, 2]]\n\t\tdf_join_raw_filter.to_csv(f"experiments/output/{exp}/filter_raw/{encoder}_{group_id}.csv", sep=";", decimal=",", index_label="Id")\n\t\tdf_join_raw_filter '